In [ ]:
import torch
import torch.nn as nn
from torch import Tensor

embeding时，我们将embeding分量初始化为均值为0，标准差为 (d_model)^(-1/2)。每一个分量都满足均值为0，标准差为(d_model)^(-1/2)的正太分布，每一个embedding向量的范数是1。
E(||x||^2)=\sum_{i=1}^{d_model}E{(x_i)^2}
E((x_i)^2) = E(((x_i)-E(x))^2) = Var(x), 因为 E(x_i) = 0 和 Var(x_i)= d_model^(-1)，所以E((x_i)^2) = d_model^(-1)。
所以E(||x||^2)=1


In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size: int, d_model: int) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        # 可训练参数：每行是一个 token 的向量
        self.weight = nn.Parameter(torch.empty(vocab_size, d_model))
        nn.init.normal_(self.weight, mean=0.0, std=d_model ** -0.5)

    def forward(self, token_ids: Tensor) -> Tensor:
        """
        token_ids: (B, L), dtype=torch.long
        返回: (B, L, D)
        """
                # (B, L) -> (B, L, D)
        x = self.weight[token_ids]          # 花式索引即查表
        if self.scale:
            x = x * math.sqrt(self.d_model) 
        return x
        
    

In [ ]:
位置编码
我们将每一个token的每两个特征设置为一组，设token是整个序列中的第i个token，该token的第j组。
则有：p_{i,2j} = sin(\frac{i}{10000^{2j/d}})
p_{i,2j+1} = cos(\frac{i}{10000^{2j/d}})
\(i\) 决定当前采样位置，\(j\) 决定使用哪一种频率。固定 \(j\) 沿 \(i\) 观察时频率不变；切换 \(j\) 时，频率才会改变。

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        # 构造分母
        index = torch.arange(0, d_model, 2).float()
        scale = -math.log(10000.0) / d_model
        div_term = torch.exp(index * scale)
        position = torch.arange(max_len).unsqueeze(1)
        pe[:,0::2] = torch.sin(div_term * position)
        pe[:,1::2] = torch.cos(div_term * position)
        self.register_buffer("pe",pe.unsqueeze(0))

    def forward(self, x: Tensor, start_pos: int = 0) -> Tensor:
        L = x.shape(1)
        return x + self.pe[:,start_pos:start_pos+L,:]
        

    
    